# State Farm Distracted Driver Detection Dataset Analysis
---
You can find the original dataset [here](https://www.kaggle.com/c/state-farm-distracted-driver-detection).

Fisrt, let's install the required packages.

In [9]:
%pip install pandas plotly

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.7 MB 11.8 MB/s eta 0:00:01
   ------------------ --------------------- 4.5/9.7 MB 12.2 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.7 MB 11.8 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 12.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 11.4 MB/s  0:00:00

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   --------------

## Total number of labels

In [7]:
import pandas as pd
from config.paths.state_farm_ddd_paths import LABELS_CSV

df_labels = pd.read_csv(LABELS_CSV)

print(f"Total labels: {len(df_labels)}")
display(df_labels)

Total labels: 10


,label,meaning
0,c0,safe driving
1,c1,texting - right
2,c2,talking on the phone - right
3,c3,texting - left
4,c4,talking on the phone - left
5,c5,operating the radio
6,c6,drinking
7,c7,reaching behind
8,c8,hair and makeup
9,c9,talking to passenger


## Number of images (total and per label)

In [28]:
def get_class_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate the number and percentage of images per class.

    Args:
        df: DataFrame containing a 'classname' column.

    Returns:
        DataFrame with each class, its image count, and its percentage.
    """
    total_images = len(df)

    # Calculate the number of images per class
    label_summary = (
        df["classname"]
        .value_counts()
        .rename_axis("label")
        .reset_index(name="image_count")
    )

    # Sort classes by image count
    label_summary = label_summary.sort_values(
        "image_count",
        ascending=True
    )

    # Calculate percentages
    label_summary["percentage"] = (
        label_summary["image_count"] / total_images * 100
    ).round(2)

    # Display class statistics
    display(label_summary.sort_values("image_count", ascending=False))

    return label_summary

In [29]:
def get_total_images(df: pd.DataFrame) -> int:
    """
    Calculate the total number of images.

    Args:
        df: DataFrame containing the dataset records.

    Returns:
        Total number of images.
    """
    total_images = len(df)

    print(f"Total number of images: {total_images}")

    return total_images

In [18]:
import plotly.express as px

def plot_class_distribution(df: pd.DataFrame) -> None:
    """
    Create an interactive horizontal bar chart showing class distribution.

    Args:
        df: DataFrame containing 'label', 'image_count', and 'percentage' columns.
    """
    # Create the horizontal bar chart
    fig = px.bar(
        df,
        x="image_count",
        y="label",
        orientation="h",
        text="image_count",
        custom_data=["percentage"],
        title="Class Distribution",
        labels={
            "label": "Class",
            "image_count": "Number of images",
        },
    )

    # Configure bar labels and hover information
    fig.update_traces(
        textposition="outside",
        hovertemplate=(
            "<b>Class:</b> %{y}<br>"
            "<b>Images:</b> %{x}<br>"
            "<b>Percentage:</b> %{customdata[0]:.2f}%"
            "<extra></extra>"
        ),
    )

    # Configure chart layout
    fig.update_layout(
        template="plotly_white",
        width=900,
        height=550,
        xaxis_title="Number of images",
        yaxis_title="Class",
        showlegend=False,
    )

    # Display the chart
    fig.show()

In [39]:

import numpy as np


def evaluate_class_balance(df_summary):
    """
    Evaluate class balance using the imbalance ratio,
    coefficient of variation, and Gini coefficient.

    Args:
        df_summary: DataFrame containing 'label' and 'image_count'.

    Returns:
        Dictionary containing the class balance metrics.
    """
    counts = df_summary["image_count"].to_numpy(dtype=float)

    if len(counts) == 0:
        raise ValueError("The class summary cannot be empty.")

    if not np.isfinite(counts).all() or (counts < 0).any():
        raise ValueError("Image counts must be finite and non-negative.")

    max_count = counts.max()
    min_count = counts.min()
    mean_count = counts.mean()
    std_count = counts.std(ddof=0)

    # Calculate the imbalance ratio
    imbalance_ratio = (
        max_count / min_count if min_count > 0 else float("inf")
    )

    # Calculate the coefficient of variation as a percentage
    coefficient_of_variation = (
        std_count / mean_count * 100 if mean_count > 0 else 0.0
    )

    # Calculate the Gini coefficient
    sorted_counts = np.sort(counts)
    n = len(sorted_counts)

    if sorted_counts.sum() > 0:
        gini_coefficient = (
            2 * np.sum(
                np.arange(1, n + 1) * sorted_counts
            ) / (n * sorted_counts.sum())
            - (n + 1) / n
        )
    else:
        gini_coefficient = 0.0

    # Display results
    print(f"Largest class: {max_count:.0f} images")
    print(f"Smallest class: {min_count:.0f} images")

    print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")
    print(
        f"Coefficient of variation: "
        f"{coefficient_of_variation:.2f}%"
    )
    print(f"Gini coefficient: {gini_coefficient:.4f}")

    # Provide an interpretation based on the imbalance ratio
    if imbalance_ratio <= 1.5:
        print("The dataset is fairly balanced.")
    elif imbalance_ratio <= 2:
        print("The dataset has mild class imbalance.")
    elif imbalance_ratio <= 5:
        print("The dataset has moderate class imbalance.")
    else:
        print("The dataset has substantial class imbalance.")

    return {
        "imbalance_ratio": imbalance_ratio,
        "coefficient_of_variation": coefficient_of_variation,
        "gini_coefficient": gini_coefficient,
    }


### Training set

In [40]:
from config.paths.state_farm_ddd_paths import TRAIN_RAW_CSV

# Load the training dataset
df_train = pd.read_csv(TRAIN_RAW_CSV)

# Calculate class distribution
label_summary = get_class_distribution(df_train)

# Calculate total number of images
total_images = get_total_images(df_train)

# Create and display the class distribution plot
plot_class_distribution(label_summary)

# Evaluate class balance
imbalance_ratio = evaluate_class_balance(label_summary)


Class Distribution:


,label,image_count,percentage
0,c0,2489,11.10
1,c1,2267,10.11
2,c2,2317,10.33
3,c3,2346,10.46
4,c4,2326,10.37
5,c5,2312,10.31
6,c6,2325,10.37
7,c7,2002,8.93
8,c8,1911,8.52
9,c9,2129,9.49


Total number of images: 22424


Largest class: 2489 images
Smallest class: 1911 images
Imbalance ratio: 1.30:1
Coefficient of variation: 7.42%
Gini coefficient: 0.0391
The dataset is fairly balanced.


### Test set

In [41]:
from config.paths.state_farm_ddd_paths import TEST_RAW_CSV

# Load the test dataset
df_test = pd.read_csv(TEST_RAW_CSV)

# Calculate class distribution
label_summary = get_class_distribution(df_test)

# Calculate total number of images
total_images = get_total_images(df_test)

# Create and display the class distribution plot
plot_class_distribution(label_summary)

# Evaluate class balance
imbalance_ratio = evaluate_class_balance(label_summary)


Class Distribution:


,label,image_count,percentage
0,c0,127,12.7
1,c1,109,10.9
2,c2,108,10.8
3,c3,98,9.8
4,c4,108,10.8
5,c5,95,9.5
6,c6,109,10.9
7,c7,91,9.1
8,c8,78,7.8
9,c9,77,7.7


Total number of images: 1000


Largest class: 127 images
Smallest class: 77 images
Imbalance ratio: 1.65:1
Coefficient of variation: 14.64%
Gini coefficient: 0.0806
The dataset has mild class imbalance.


## Number of unique drivers (training)

In [42]:
import numpy as np
# Calculate the total number of unique drivers
total_drivers = df_train["subject"].nunique()

print(f"Total number of drivers: {total_drivers}")


# Calculate the number of images contributed by each driver in each class
driver_participation = (
    df_train.groupby(["classname", "subject"])
    .size()
    .reset_index(name="image_count")
    .sort_values(["classname", "subject"])
)


# Calculate the percentage of images contributed by each driver within their class
driver_participation["class_percentage"] = (
    driver_participation["image_count"]
    / driver_participation.groupby("classname")["image_count"].transform("sum")
    * 100
).round(2)


# Calculate participation balance metrics for each class
def calculate_gini(values):
    """Calculate the Gini coefficient for non-negative values."""
    values = np.sort(np.asarray(values, dtype=float))
    n = len(values)

    if n == 0 or values.sum() == 0:
        return 0.0

    return float(
        2 * np.sum(np.arange(1, n + 1) * values)
        / (n * values.sum())
        - (n + 1) / n
    )


class_participation_balance = (
    driver_participation.groupby("classname")["image_count"]
    .agg(
        driver_count="count",
        min_images="min",
        max_images="max",
        mean_images="mean",
        std_images=lambda values: values.std(ddof=0),
    )
    .reset_index()
)


# Calculate the participation ratio
class_participation_balance["participation_ratio"] = (
    class_participation_balance["max_images"]
    / class_participation_balance["min_images"]
).round(2)


# Calculate the coefficient of variation as a percentage
class_participation_balance["coefficient_of_variation"] = (
    class_participation_balance["std_images"]
    / class_participation_balance["mean_images"]
    * 100
).round(2)


# Calculate the Gini coefficient
gini_values = (
    driver_participation.groupby("classname")["image_count"]
    .apply(calculate_gini)
    .rename("gini_coefficient")
)

class_participation_balance = class_participation_balance.merge(
    gini_values,
    on="classname",
)

class_participation_balance["gini_coefficient"] = (
    class_participation_balance["gini_coefficient"].round(4)
)


# Display the participation balance summary
print("\nDriver Participation Balance by Class:")
display(class_participation_balance)


# Display individual driver participation for each class
for class_name, class_data in driver_participation.groupby("classname"):
    print(f"\nClass: {class_name}")
    print(
        f"Number of participating drivers: "
        f"{class_data['subject'].nunique()}"
    )

    display(
        class_data[
            ["subject", "image_count", "class_percentage"]
        ].reset_index(drop=True)
    )

Total number of drivers: 26

Driver Participation Balance by Class:


,classname,driver_count,min_images,max_images,mean_images,std_images,participation_ratio,coefficient_of_variation,gini_coefficient
0,c0,26,59,182,95.730769,29.170003,3.08,30.47,0.1633
1,c1,26,45,131,87.192308,22.403673,2.91,25.69,0.1408
2,c2,26,36,130,89.115385,23.554074,3.61,26.43,0.1462
3,c3,26,31,131,90.230769,24.344179,4.23,26.98,0.1482
4,c4,26,34,132,89.461538,23.376896,3.88,26.13,0.1450
5,c5,26,6,131,88.923077,26.634164,21.83,29.95,0.1591
6,c6,26,35,131,89.423077,23.632073,3.74,26.43,0.1468
7,c7,26,2,101,77.000000,18.904619,50.50,24.55,0.1163
8,c8,26,21,114,73.500000,20.954621,5.43,28.51,0.1588
9,c9,26,51,131,81.884615,23.578554,2.57,28.79,0.1580



Class: c0
Number of participating drivers: 26


,subject,image_count,class_percentage
0,p002,76,3.05
1,p012,84,3.37
2,p014,100,4.02
3,p015,79,3.17
4,p016,111,4.46
5,p021,135,5.42
6,p022,129,5.18
7,p024,130,5.22
8,p026,130,5.22
9,p035,94,3.78



Class: c1
Number of participating drivers: 26


...

,subject,image_count,class_percentage
0,p002,72,3.60
1,p012,72,3.60
2,p014,77,3.85
3,p015,81,4.05
4,p016,101,5.04
5,p021,98,4.90
6,p022,98,4.90
7,p024,101,5.04
8,p026,97,4.85
9,p035,87,4.35



Class: c8
Number of participating drivers: 26


,subject,image_count,class_percentage
0,p002,44,2.30
1,p012,62,3.24
2,p014,38,1.99
3,p015,86,4.50
4,p016,99,5.18
5,p021,99,5.18
6,p022,98,5.13
7,p024,99,5.18
8,p026,97,5.08
9,p035,56,2.93



Class: c9
Number of participating drivers: 26


,subject,image_count,class_percentage
0,p002,51,2.40
1,p012,62,2.91
2,p014,52,2.44
3,p015,61,2.87
4,p016,120,5.64
5,p021,131,6.15
6,p022,131,6.15
7,p024,120,5.64
8,p026,98,4.60
9,p035,81,3.80
